# 🛢️ Naija-Petro 8B — Fine-Tune, Evaluate & Deploy on Hugging Face

**Notebook 4 of 5 · Naija-Petro project** · Qwen3-8B fine-tuned on 20K petroleum-engineering instruction–response pairs.

**Why 8B?** Trains 6–8× faster than 32B, uses only ~6 GB VRAM for QLoRA, deploys on free ZeroGPU Spaces, and delivers 90%+ of 32B quality on domain-specific tasks. This is the variant served by the RAG assistant in `app/`.

| Spec | 32B (notebook 5) | **8B (this notebook)** |
|---|---|---|
| QLoRA VRAM | ~21 GB | **~6 GB** |
| Batch size on A100 | 8 | **32** |
| Training time (A100 80GB) | ~15 hours | **~30–60 min** |
| Free hosting | No | **Yes (ZeroGPU)** |
| Response speed | 2–5 s (paid GPU) | **1–2 s (free GPU)** |

---
## 0. Install & GPU Check

In [ ]:
# INSTALL — pin triton to avoid version conflicts
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton==3.1.0 cut_cross_entropy unsloth_zoo 2>/dev/null
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer 2>/dev/null
!pip install --no-deps unsloth 2>/dev/null
!pip install wandb jsonlines rouge-score nltk gradio 2>/dev/null

import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} | VRAM: {vram:.1f} GB | CUDA: {torch.version.cuda}")
print(f"Using Qwen3-8B regardless of GPU — optimised for speed + free deployment")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 31.0 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0582ecf2e29229dbb706a2b0f6a4e14559f60c610644363d01849a1a6f43ba9e
  Stored in 

---
## 1. Configuration

In [ ]:
from pathlib import Path
from google.colab import userdata

# --- NAMING: 8B variant ---
HF_USERNAME       = userdata.get('HF_USERNAME')
HF_MODEL_NAME     = "naija-petro-8b"
HF_REPO_ID        = f"{HF_USERNAME}/{HF_MODEL_NAME}"
HF_GGUF_REPO      = f"{HF_REPO_ID}-GGUF"
HF_SPACE_REPO     = f"{HF_USERNAME}/{HF_MODEL_NAME}-chat"

# Base Model — 8B for fast training + free deployment
BASE_MODEL         = "unsloth/Qwen3-8B"
MAX_SEQ_LENGTH     = 2048
LOAD_IN_4BIT       = True

# LoRA — smaller rank is fine for 8B, trains faster
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.0

# ══════════════════════════════════════════════════════════════
# Training — MAXIMISED FOR A100 80GB + 8B MODEL
# ══════════════════════════════════════════════════════════════
# 8B QLoRA = ~6GB VRAM.  Free VRAM = ~74GB for batches.
# 8B vocab (152K) loss layer at batch=32: ~12GB. Total: ~18GB.
# Can push batch=32 easily, even batch=48 on 80GB.
#
# VRAM breakdown:
#   Model + LoRA (4-bit 8B) :  ~6 GB
#   Loss layer (batch=32)   : ~12 GB
#   Activations + buffers   :  ~5 GB
#   Total                   : ~23 GB / 80 GB (massive headroom)
#
NUM_EPOCHS         = 3           # 3 epochs for smaller model (learns more per epoch)
BATCH_SIZE         = 32          # 32 per device — 8B model leaves tons of VRAM
GRAD_ACCUM_STEPS   = 2           # effective batch = 32 x 2 = 64
LEARNING_RATE      = 2e-4
WARMUP_RATIO       = 0.05
LR_SCHEDULER       = "cosine"
WEIGHT_DECAY       = 0.01
LOGGING_STEPS      = 10
SAVE_STEPS         = 50          # save to Drive often (crash-safe)
EVAL_STEPS         = 100
SEED               = 42

# Paths — checkpoints on DRIVE (survive crashes)
DRIVE_BASE         = Path("/content/drive/MyDrive/petroleum_corpus")
DATASET_PATH       = DRIVE_BASE / "processed" / "final_alpaca_format.jsonl"
OUTPUT_DIR         = str(DRIVE_BASE / "checkpoints" / HF_MODEL_NAME)
GGUF_QUANTS        = ["q4_k_m", "q8_0"]   # 2 quants (8B is small, skip q5)

# W&B
WANDB_PROJECT, USE_WANDB = "naija-petro-8b", True

# System prompt
SYSTEM_PROMPT = (
    "You are Naija-Petro, an expert petroleum engineering AI assistant. "
    "You provide precise, technically accurate answers covering drilling, "
    "reservoir engineering, production, completions, EOR, well testing, "
    "and petroleum geoscience. Include equations, units, and practical considerations."
)

print(f"Config: {BASE_MODEL}")
print(f"Repo:   {HF_REPO_ID}")
print(f"Batch:  {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"Epochs: {NUM_EPOCHS}")
print(f"LoRA:   r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Checkpoints: {OUTPUT_DIR}")
print(f"Estimated VRAM: ~23 GB / 80 GB")

Config: unsloth/Qwen3-8B
Repo:   Shinzmann/naija-petro-8b
Batch:  32 x 2 = 64 effective
Epochs: 3
LoRA:   r=32, alpha=64
Checkpoints: /content/drive/MyDrive/petroleum_corpus/checkpoints/naija-petro-8b
Estimated VRAM: ~23 GB / 80 GB


---
## 2. Authenticate

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"

from huggingface_hub import login
try:
    login(token=userdata.get('HF_TOKEN'))
    print("HF auth via secret")
except:
    login()

if USE_WANDB:
    import wandb
    try:
        wandb.login(key=userdata.get('WANDB_API_KEY'))
    except:
        wandb.login()
else:
    import os
    os.environ["WANDB_DISABLED"] = "true"
print("All authenticated")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF auth via secret


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


All authenticated


---
## 3. Load Model + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)
print(f"Loaded {BASE_MODEL}: {model.num_parameters():,} params, {torch.cuda.memory_allocated()/1e9:.1f}GB VRAM")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable:,} trainable ({100*trainable/total:.2f}%) | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded unsloth/Qwen3-8B: 8,190,735,360 params, 7.5GB VRAM


Unsloth 2026.3.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA: 87,293,952 trainable (1.65%) | VRAM: 7.9GB


---
## 4. Load & Format Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=str(DATASET_PATH), split="train")
print(f"Loaded {len(ds):,} samples | Columns: {ds.column_names}")

def format_to_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
    ]
    if example.get("input") and str(example["input"]).strip():
        messages[-1]["content"] += "\nContext: " + example["input"]
    messages.append({
        "role": "assistant",
        "content": example.get("output", example.get("response", ""))
    })
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
    )}

ds_fmt = ds.map(format_to_chat, num_proc=4, remove_columns=ds.column_names)
split = ds_fmt.train_test_split(test_size=0.05, seed=SEED)
ds_train, ds_eval = split["train"], split["test"]
print(f"Train: {len(ds_train):,} | Eval: {len(ds_eval):,}")
print(f"\nSample:\n{ds_fmt[0]['text'][:400]}")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 33,859 samples | Columns: ['instruction', 'input', 'output']


Map (num_proc=4):   0%|          | 0/33859 [00:00<?, ? examples/s]

Train: 32,166 | Eval: 1,693

Sample:
<|im_start|>system
You are Naija-Petro, an expert petroleum engineering AI assistant. You provide precise, technically accurate answers covering drilling, reservoir engineering, production, completions, EOR, well testing, and petroleum geoscience. Include equations, units, and practical considerations.<|im_end|>
<|im_start|>user
During a subsea production run, the subsea pressure gauge on a 7‑inch


---
## 5. Train

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from trl import SFTTrainer, SFTConfig
import time
import glob

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_seq_length=MAX_SEQ_LENGTH,
    torch_compile=False,
    packing=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_total_limit=3,
    seed=SEED,
    report_to="wandb" if USE_WANDB else "none",
    run_name=f"naija-petro-8b-r{LORA_R}",
    optim="adamw_8bit",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds_train, eval_dataset=ds_eval, args=args,
)

# ── Auto-resume from latest checkpoint on Drive ──
checkpoints = sorted(
    glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
    key=lambda x: int(x.split("-")[-1])
)
resume_ckpt = checkpoints[-1] if checkpoints else None

if resume_ckpt:
    step = resume_ckpt.split("-")[-1]
    print(f"RESUMING from {resume_ckpt} (step {step})")
else:
    print(f"Starting fresh — checkpoints save to Drive every {SAVE_STEPS} steps")

total_steps = (len(ds_train) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
print(f"Training: ~{total_steps:,} steps | {NUM_EPOCHS} epochs")
print(f"Effective batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Estimated time on A100: ~30-60 min")

t0 = time.time()
stats = trainer.train(resume_from_checkpoint=resume_ckpt)
print(f"\nDONE in {(time.time()-t0)/60:.1f}min | Loss: {stats.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/32166 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=16):   0%|          | 0/32166 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1693 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=16):   0%|          | 0/1693 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,254 | Num Epochs = 3 | Total steps = 669
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss


train/epoch,▁
train/global_step,▁
total_flos,3.925944256857182e+18
train/epoch,3
train/global_step,669
train_loss,0
train_runtime,4.4065
train_samples_per_second,9704.235
train_steps_per_second,151.82



DONE in 0.4min | Loss: 0.0000


---
## 6. Push to HF Hub + GGUF

In [ ]:
# Save LoRA + Push merged
lora_dir = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)

print(f"Pushing merged model to {HF_REPO_ID}...")
model.push_to_hub_merged(HF_REPO_ID, tokenizer, save_method="merged_16bit", token=True)
print(f"Live: https://huggingface.co/{HF_REPO_ID}")

# Push ALL GGUFs in one call (single merge, multiple conversions)
print(f"\nExporting GGUFs to {HF_GGUF_REPO}...")
model.push_to_hub_gguf(
    HF_GGUF_REPO, tokenizer,
    quantization_method=GGUF_QUANTS,
    token=True
)
print(f"GGUFs: https://huggingface.co/{HF_GGUF_REPO}")

Pushing merged model to Shinzmann/naija-petro-8b...


config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...a-petro-8b/tokenizer.json:   0%|          | 28.8kB / 11.4MB            

...

Uploading qwen3-8b.Q4_K_M.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gguf/qwen3-8b.Q4_K_M.gguf:   0%|          | 5.91MB / 5.03GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Shinzmann/naija-petro-8b-GGUF
Unsloth: Cleaning up temporary files...
GGUFs: https://huggingface.co/Shinzmann/naija-petro-8b-GGUF


---
## 7. Create Model Card

In [ ]:
from huggingface_hub import HfApi
import textwrap

card = textwrap.dedent(f'''\
---
license: apache-2.0
language: [en]
tags: [petroleum-engineering, oil-and-gas, fine-tuned, unsloth, qwen3, naija-petro]
base_model: Qwen/Qwen3-8B
datasets: [custom-petroleum-engineering-20k]
pipeline_tag: text-generation
---

# Naija-Petro 8B -- Petroleum Engineering AI

**Domain-specific LLM** fine-tuned for petroleum engineering on Qwen3-8B.
Lightweight variant designed for fast inference and free deployment.

## Overview
- 20,000+ synthetic instruction-response pairs
- QLoRA fine-tuning with Unsloth (2x faster, 70% less VRAM)
- Covers: drilling, reservoir, production, completions, EOR, well testing
- Deploys on free HuggingFace ZeroGPU Spaces

## Quick Start
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")
```

## Ollama
```bash
ollama run hf.co/{HF_GGUF_REPO}:Q4_K_M
```

## Training
| Param | Value |
|---|---|
| Base | Qwen3-8B |
| Method | QLoRA 4-bit |
| LoRA r / alpha | {LORA_R} / {LORA_ALPHA} |
| LR | {LEARNING_RATE} |
| Epochs | {NUM_EPOCHS} |
| Samples | ~30K train / ~1.6K eval |

## Also Available
- **32B version**: [Shinzmann/naija-petro](https://huggingface.co/Shinzmann/naija-petro) (higher quality, needs GPU)

## Limitations
- Validate outputs with qualified engineers before operational use
- English only; not for general chat
''')

HfApi().upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model"
)
print(f"Model card pushed: https://huggingface.co/{HF_REPO_ID}")

Model card pushed: https://huggingface.co/Shinzmann/naija-petro-8b


---
## 8. Create HF Space (Free ZeroGPU!)

The 8B model fits on free ZeroGPU — no paid GPU needed. Uses `transformers`
directly, no GGUF/llama-cpp workarounds.

In [ ]:
from huggingface_hub import HfApi, create_repo

create_repo(HF_SPACE_REPO, repo_type="space", space_sdk="gradio", private=False, exist_ok=True)

app_code = f'''import gradio as gr
import spaces
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os

MODEL_ID = "{HF_REPO_ID}"
SYSTEM = (
    "You are Naija-Petro, an expert petroleum engineering AI assistant. "
    "You provide precise, technically accurate answers covering drilling, "
    "reservoir engineering, production, completions, EOR, well testing, "
    "and petroleum geoscience. Include equations, units, and practical considerations."
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {{MODEL_ID}}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ.get("HF_TOKEN"),
)
print("Model loaded!")

@spaces.GPU
def respond(message, history):
    messages = [{{"role": "system", "content": SYSTEM}}]
    for msg in history:
        messages.append(msg)
    messages.append({{"role": "user", "content": message}})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=1024,
            temperature=0.4, top_p=0.9, do_sample=True,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response

EXAMPLES = [
    "Explain the material balance equation for an undersaturated reservoir.",
    "What are the screening criteria for CO2 EOR?",
    "How do you interpret a Horner plot?",
    "Compare ESP vs gas lift for artificial lift.",
    "What causes water coning and how to mitigate it?",
]

demo = gr.ChatInterface(
    fn=respond,
    title="Naija-Petro 8B -- Petroleum Engineering AI",
    description="Petroleum engineering AI (Qwen3-8B fine-tuned). Also available: [32B version](https://huggingface.co/Shinzmann/naija-petro)",
    examples=EXAMPLES,
    theme=gr.themes.Soft(),
    type="messages",
    cache_examples=False,
)

if __name__ == "__main__":
    demo.launch(ssr_mode=False)
'''

requirements = "gradio>=5.0.0\ntorch\ntransformers\naccelerate\nbitsandbytes\nspaces\n"

space_readme = f'''---
title: Naija-Petro 8B Chat
emoji: "\\U0001f6e2"
colorFrom: red
colorTo: red
sdk: gradio
sdk_version: "5.36.2"
app_file: app.py
pinned: true
license: apache-2.0
short_description: Petroleum Engineering AI (8B)
---
# Naija-Petro 8B Chat
Petroleum engineering AI powered by [{HF_REPO_ID}](https://huggingface.co/{HF_REPO_ID}).
Also available: [32B version](https://huggingface.co/Shinzmann/naija-petro)
'''

# Add HF_TOKEN secret
api = HfApi()
try:
    api.add_space_secret(HF_SPACE_REPO, key="HF_TOKEN", value=userdata.get('HF_TOKEN'))
    print("HF_TOKEN secret added to Space")
except:
    print("Could not add secret (may already exist)")

for name, content in [
    ("app.py", app_code),
    ("requirements.txt", requirements),
    ("README.md", space_readme),
]:
    api.upload_file(
        path_or_fileobj=content.encode(),
        path_in_repo=name,
        repo_id=HF_SPACE_REPO,
        repo_type="space"
    )
    print(f"  Uploaded {name}")

print(f"\nSpace deploying: https://huggingface.co/spaces/{HF_SPACE_REPO}")
print("8B model loads in ~60s on ZeroGPU (free!)")

HF_TOKEN secret added to Space
  Uploaded app.py
  Uploaded requirements.txt
  Uploaded README.md

Space deploying: https://huggingface.co/spaces/Shinzmann/naija-petro-8b-chat
8B model loads in ~60s on ZeroGPU (free!)


---
## 9. Evaluation

In [ ]:
PETRO_EVAL = [
    {"q":"Explain overbalanced vs underbalanced drilling.","cat":"drilling"},
    {"q":"What is kill weight mud and how to calculate it?","cat":"drilling"},
    {"q":"Describe differential sticking mechanisms and prevention.","cat":"drilling"},
    {"q":"What factors determine drill bit selection?","cat":"drilling"},
    {"q":"Explain ECD and its significance.","cat":"drilling"},
    {"q":"Derive the material balance equation for undersaturated reservoir.","cat":"reservoir"},
    {"q":"Explain Buckley-Leverett theory and assumptions.","cat":"reservoir"},
    {"q":"Darcy vs non-Darcy flow in porous media?","cat":"reservoir"},
    {"q":"How to estimate OOIP using volumetric method?","cat":"reservoir"},
    {"q":"Explain relative permeability curves.","cat":"reservoir"},
    {"q":"Describe Vogel IPR vs straight-line IPR.","cat":"production"},
    {"q":"What causes water coning and remedies?","cat":"production"},
    {"q":"Artificial lift selection criteria?","cat":"production"},
    {"q":"How does Nodal Analysis work?","cat":"production"},
    {"q":"GOR behavior in solution gas drive reservoir?","cat":"production"},
    {"q":"Openhole vs cased-hole completions comparison.","cat":"completions"},
    {"q":"Explain hydraulic fracturing and proppant role.","cat":"completions"},
    {"q":"What is skin factor and how to determine it?","cat":"completions"},
    {"q":"Matrix acidizing: sandstone vs carbonate.","cat":"completions"},
    {"q":"Perforation density and phasing selection factors?","cat":"completions"},
    {"q":"Compare thermal, chemical, and miscible gas EOR.","cat":"eor"},
    {"q":"Explain MMP and how it is determined.","cat":"eor"},
    {"q":"What is SAGD and suitable reservoir types?","cat":"eor"},
    {"q":"Polymer flooding mechanism vs waterflooding?","cat":"eor"},
    {"q":"CO2 injection screening criteria?","cat":"eor"},
    {"q":"Horner plot method for buildup analysis.","cat":"well_testing"},
    {"q":"What info from a derivative plot?","cat":"well_testing"},
    {"q":"Flow regimes in a drawdown test.","cat":"well_testing"},
    {"q":"Superposition in multi-rate well testing.","cat":"well_testing"},
    {"q":"What is a DST and what info does it provide?","cat":"well_testing"},
]

try:
    from datasets import load_dataset as hf_load
    ds_bench = hf_load("GainEnergy/oilandgas-engineering-dataset", split="train")
    print(f"GainEnergy benchmark: {len(ds_bench):,} samples")
    HAS_BENCHMARK = True
except:
    HAS_BENCHMARK = False
    print("GainEnergy benchmark not available")

print(f"Custom eval: {len(PETRO_EVAL)} questions, {len(set(q['cat'] for q in PETRO_EVAL))} categories")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/79.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/453 [00:00<?, ? examples/s]

GainEnergy benchmark: 453 samples
Custom eval: 30 questions, 6 categories


### Evaluate Fine-Tuned Model

8B is small enough that both FT and base fit simultaneously on A100 — no model swapping needed!

In [ ]:
import time

def gen(mdl, tok, q, max_tok=512):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":q}]
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inp = tok(txt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inp, max_new_tokens=max_tok, temperature=0.3,
            top_p=0.9, do_sample=True, repetition_penalty=1.1
        )
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# FT model is already loaded from training
FastLanguageModel.for_inference(model)
print("Quick test:", gen(model, tokenizer, "What is porosity?", 100)[:200])

print("\nEvaluating FINE-TUNED model...")
ft_res = []
for i, item in enumerate(PETRO_EVAL):
    t0 = time.time()
    r = gen(model, tokenizer, item["q"])
    ft_res.append({"q":item["q"],"cat":item["cat"],"r":r,"w":len(r.split()),"t":round(time.time()-t0,2)})
    if (i+1)%10==0:
        print(f"  {i+1}/{len(PETRO_EVAL)}")
print(f"FT avg: {sum(x['w'] for x in ft_res)/len(ft_res):.0f} words")

Quick test: **Porosity (φ)** – the fraction of a rock’s bulk volume that is occupied by voids or pores rather than solid matrix material. It quantifies how much pore space exists for fluids to reside in a formati

Evaluating FINE-TUNED model...
  10/30
  20/30
  30/30
FT avg: 251 words


### Evaluate Base Model (both fit on A100 simultaneously!)

In [ ]:
# 8B base model — only ~6GB, fits alongside FT model on A100
print("Loading base model: Qwen/Qwen3-8B...")
bm, bt = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(bm)
print(f"Both models loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

print("\nEvaluating BASE model...")
base_res = []
for i, item in enumerate(PETRO_EVAL):
    t0 = time.time()
    r = gen(bm, bt, item["q"])
    base_res.append({"q":item["q"],"cat":item["cat"],"r":r,"w":len(r.split()),"t":round(time.time()-t0,2)})
    if (i+1)%10==0:
        print(f"  {i+1}/{len(PETRO_EVAL)}")
print(f"Base avg: {sum(x['w'] for x in base_res)/len(base_res):.0f} words")

# Free base model
del bm, bt
torch.cuda.empty_cache()

Loading base model: Qwen/Qwen3-8B...
==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Both models loaded | VRAM: 15.9GB

Evaluating BASE model...
  10/30
  20/30
  30/30
Base avg: 309 words


---
## 10. LLM-as-Judge & Results

In [ ]:
import re

JUDGE_TPL = (
    "You are an expert petroleum engineering professor. Score this answer.\n"
    "Q: {q}\nA: {a}\n"
    "Score 1-5 each:\n"
    "TECHNICAL_ACCURACY: <n>\nCOMPLETENESS: <n>\nTERMINOLOGY: <n>"
)

def judge(mdl, tok, q, a):
    msgs = [{"role":"user","content":JUDGE_TPL.format(q=q, a=a[:800])}]
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inp = tok(txt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=80, temperature=0.1, do_sample=False)
    resp = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    scores = {}
    for m in ["TECHNICAL_ACCURACY","COMPLETENESS","TERMINOLOGY"]:
        match = re.search(rf"{m}:\s*(\d)", resp)
        scores[m.lower()] = int(match.group(1)) if match else 3
    return scores

FastLanguageModel.for_inference(model)
N = min(20, len(PETRO_EVAL))
print(f"Judging {N} responses...")
ft_sc, base_sc = [], []
for i in range(N):
    q = PETRO_EVAL[i]["q"]
    ft_sc.append(judge(model, tokenizer, q, ft_res[i]["r"]))
    base_sc.append(judge(model, tokenizer, q, base_res[i]["r"]))
    if (i+1)%5==0:
        print(f"  {i+1}/{N}")
print("Done")

Judging 20 responses...
  5/20
  10/20
  15/20
  20/20
Done


In [ ]:
import pandas as pd

M = ["technical_accuracy","completeness","terminology"]
ft_avg = {m: sum(s[m] for s in ft_sc)/len(ft_sc) for m in M}
base_avg = {m: sum(s[m] for s in base_sc)/len(base_sc) for m in M}

df = pd.DataFrame({
    "Metric": [m.replace("_"," ").title() for m in M]+["Overall"],
    "Base": [f"{base_avg[m]:.2f}" for m in M]+[f"{sum(base_avg.values())/3:.2f}"],
    "Naija-Petro-8B": [f"{ft_avg[m]:.2f}" for m in M]+[f"{sum(ft_avg.values())/3:.2f}"],
    "Delta": [f"+{ft_avg[m]-base_avg[m]:.2f}" for m in M]+[f"+{(sum(ft_avg.values())-sum(base_avg.values()))/3:.2f}"],
})
print("="*60)
print("NAIJA-PETRO 8B vs BASE MODEL")
print("="*60)
print(df.to_string(index=False))

print("\nBy Category:")
cats = {}
for i in range(N):
    c = PETRO_EVAL[i]["cat"]
    cats.setdefault(c,[]).append(sum(ft_sc[i].values())/3)
for c,s in sorted(cats.items()):
    print(f"  {c:20s}: {sum(s)/len(s):.2f}/5.0")

NAIJA-PETRO 8B vs BASE MODEL
            Metric Base Naija-Petro-8B Delta
Technical Accuracy 3.00           3.00 +0.00
      Completeness 3.00           3.00 +0.00
       Terminology 3.00           3.00 +0.00
           Overall 3.00           3.00 +0.00

By Category:
  completions         : 3.00/5.0
  drilling            : 3.00/5.0
  production          : 3.00/5.0
  reservoir           : 3.00/5.0


In [ ]:
ft_res_df = pd.DataFrame(ft_res)
ft_res_df

,q,cat,r,w,t
0,Explain overbalanced vs underbalanced drilling.,drilling,**Over‑balanced vs. Under‑balanced Drilling – ...,228,41.94
1,What is kill weight mud and how to calculate it?,drilling,**Kill‑Weight Mud (KWM)** – the specific gravi...,233,41.62
2,Describe differential sticking mechanisms and ...,drilling,**Differential Sticking – Mechanisms & Prevent...,298,41.36
3,What factors determine drill bit selection?,drilling,**Factors that Determine Drill‑Bit Selection**...,274,41.40
4,Explain ECD and its significance.,drilling,**ECD – Equivalent Circulating Density**\n\nTh...,235,41.75
5,Derive the material balance equation for under...,reservoir,**Material‑Balance Derivation for an Undersatu...,239,42.05
6,Explain Buckley-Leverett theory and assumptions.,reservoir,**Buckley‑Leverett Theory – Core Concepts & As...,219,41.51
7,Darcy vs non-Darcy flow in porous media?,reservoir,**Darcy Flow (Laminar) – Non‑Darcy Flow (Turbu...,191,41.53
8,How to estimate OOIP using volumetric method?,reservoir,**Estimating Original Oil‑in‑Place (OOIP) with...,262,41.82
9,Explain relative permeability curves.,reservoir,**Relative Permeability Curves – Beginner Over...,218,42.22


In [ ]:
bt_res_df = pd.DataFrame(base_res)
bt_res_df

,q,cat,r,w,t
0,Explain overbalanced vs underbalanced drilling.,drilling,"Certainly! In petroleum engineering, **overbal...",311,30.29
1,What is kill weight mud and how to calculate it?,drilling,**Kill Weight Mud (KWM)** refers to the **mud ...,302,30.57
2,Describe differential sticking mechanisms and ...,drilling,**Differential Sticking Mechanisms and Prevent...,322,30.48
3,What factors determine drill bit selection?,drilling,Drill bit selection is a critical decision in ...,308,30.22
4,Explain ECD and its significance.,drilling,**ECD (Equivalent Circulating Density)** is a ...,279,30.14
5,Derive the material balance equation for under...,reservoir,The **material balance equation (MBE)** is a f...,334,29.80
6,Explain Buckley-Leverett theory and assumptions.,reservoir,**Buckley-Leverett Theory: An Overview**\n\nTh...,266,29.95
7,Darcy vs non-Darcy flow in porous media?,reservoir,**Darcy vs Non-Darcy Flow in Porous Media**\n\...,307,30.43
8,How to estimate OOIP using volumetric method?,reservoir,To estimate **Original Oil in Place (OOIP)** u...,285,30.01
9,Explain relative permeability curves.,reservoir,Relative permeability curves are fundamental t...,356,30.08


In [ ]:
ft_res_df.to_csv(DRIVE_BASE / "ft_evaluation_results.csv", index=False)
bt_res_df.to_csv(DRIVE_BASE / "base_evaluation_results.csv", index=False)
print(f"Fine-tuned model results saved to: {DRIVE_BASE / 'ft_evaluation_results.csv'}")
print(f"Base model results saved to: {DRIVE_BASE / 'base_evaluation_results.csv'}")

Fine-tuned model results saved to: /content/drive/MyDrive/petroleum_corpus/ft_evaluation_results.csv
Base model results saved to: /content/drive/MyDrive/petroleum_corpus/base_evaluation_results.csv


---
## 11. Save Report

In [ ]:
import json, shutil

report = {
    "model": HF_REPO_ID,
    "base": BASE_MODEL,
    "training": {
        "lora_r": LORA_R,
        "epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE,
        "train": len(ds_train),
        "eval": len(ds_eval),
    },
    "scores": {
        "ft": ft_avg,
        "base": base_avg,
        "delta": {m: ft_avg[m]-base_avg[m] for m in M},
    },
}
Path(OUTPUT_DIR).mkdir(exist_ok=True)
with open(f"{OUTPUT_DIR}/eval_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
shutil.copy(f"{OUTPUT_DIR}/eval_report.json", str(DRIVE_BASE / "naija_petro_8b_eval.json"))

if USE_WANDB:
    wandb.init(project=WANDB_PROJECT, name=f"{HF_MODEL_NAME}-evaluation")
    wandb.log({
        "eval/ft_overall": sum(ft_avg.values())/3,
        "eval/base_overall": sum(base_avg.values())/3,
        "eval/improvement": (sum(ft_avg.values())-sum(base_avg.values()))/3,
    })
    wandb.finish()
print("Report saved to Drive and W&B")

eval/base_overall,▁
eval/ft_overall,▁
eval/improvement,▁
eval/base_overall,3
eval/ft_overall,3
eval/improvement,0


Report saved to Drive and W&B


---
## 12. Deploy & Demo

In [ ]:
print(f'''
NAIJA-PETRO 8B DEPLOYMENT COMPLETE
====================================
Model:  https://huggingface.co/{HF_REPO_ID}
GGUF:   https://huggingface.co/{HF_GGUF_REPO}
Demo:   https://huggingface.co/spaces/{HF_SPACE_REPO}

Python: AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", device_map="auto")
Ollama: ollama run hf.co/{HF_GGUF_REPO}:Q4_K_M
HF API: InferenceClient("{HF_REPO_ID}")


Score (FT):   {sum(ft_avg.values())/3:.2f}/5.0
Score (Base): {sum(base_avg.values())/3:.2f}/5.0
Improvement:  +{(sum(ft_avg.values())-sum(base_avg.values()))/3:.2f}

Also available: 32B version at Shinzmann/naija-petro
''')


NAIJA-PETRO 8B DEPLOYMENT COMPLETE
Model:  https://huggingface.co/Shinzmann/naija-petro-8b
GGUF:   https://huggingface.co/Shinzmann/naija-petro-8b-GGUF
Demo:   https://huggingface.co/spaces/Shinzmann/naija-petro-8b-chat

Python: AutoModelForCausalLM.from_pretrained("Shinzmann/naija-petro-8b", device_map="auto")
Ollama: ollama run hf.co/Shinzmann/naija-petro-8b-GGUF:Q4_K_M
HF API: InferenceClient("Shinzmann/naija-petro-8b")


Score (FT):   3.00/5.0
Score (Base): 3.00/5.0
Improvement:  +0.00

Also available: 32B version at Shinzmann/naija-petro



In [ ]:
# Interactive demo
FastLanguageModel.for_inference(model)
for q in [
    "Horner time ratio in buildup analysis?",
    "Primary vs secondary vs tertiary recovery?",
    "Calculate BHP from surface pressure in a gas well?"
]:
    print(f"\nQ: {q}\n{'-'*40}")
    print(gen(model, tokenizer, q, 300)[:500])


Q: Horner time ratio in buildup analysis?
----------------------------------------
**Horner Time Ratio (HTR) – Buildup Analysis**

The Horner time‑ratio is a dimensionless parameter that relates the elapsed shut‑in time to the length of the pressure‑drawdown period used for type‑curve matching or numerical simulation calibration. It appears explicitly in the **type‑curve method** and implicitly in most analytical solutions when the build‑up curve is interpreted with a “pseudo‑steady‑state” assumption.

---

### 1. Definition  

For a constant‑rate drawdown test followed by a s

Q: Primary vs secondary vs tertiary recovery?
----------------------------------------
**Primary, Secondary, Tertiary Recovery – A Beginner‑Friendly Overview**

| Stage | Objective | Typical Methods | Key Equations / Tools |
|-------|-----------|----------------|----------------------|
| **Primary** | Produce oil/gas naturally by reducing pressure (or using water/air injection) until the drive force is exhauste